In [1]:
from pyspark.sql.functions import *

StatementMeta(, 52208f96-6661-4681-a01b-e36329753433, 3, Finished, Available, Finished)

In [6]:
from pyspark.sql.functions import (
    col, trim, when, monotonically_increasing_id
)

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

print("\nCreating dim_cause...")

dim_cause = (
    silver_df
    .select(trim(col("Cause")).alias("cause_name"))
    .dropna()
    .dropDuplicates()
    .withColumn("cause_key", monotonically_increasing_id())
    .withColumn(
        "cause_category",
        when(col("cause_name").rlike("(?i)(equipment|hardware|device|component)"), "Equipment")
        .when(col("cause_name").rlike("(?i)(weather|storm|lightning|flood|wind)"), "Environmental")
        .when(col("cause_name").rlike("(?i)(power|grid|electricity|generator)"), "Power")
        .when(col("cause_name").rlike("(?i)(human|error|mistake|configuration)"), "Human")
        .when(col("cause_name").rlike("(?i)(network|congestion|capacity|traffic|overload)"), "Capacity")
        .when(col("cause_name").rlike("(?i)(fiber|cable|link)"), "Infrastructure")
        .otherwise("Other")
    )
    .select("cause_key", "cause_name", "cause_category")
)

(
    dim_cause
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lh_Gold_Telecom.dbo.dim_cause")
)

print(f"✓ dim_cause created with {dim_cause.count()} unique causes")

dim_cause.show(100, truncate=False)


StatementMeta(, 52208f96-6661-4681-a01b-e36329753433, 8, Finished, Available, Finished)


Creating dim_cause...
✓ dim_cause created with 4 unique causes
+---------+----------+--------------+
|cause_key|cause_name|cause_category|
+---------+----------+--------------+
|0        |WEATHER   |Environmental |
|1        |FIBER     |Infrastructure|
|2        |POWER     |Power         |
|3        |HARDWARE  |Equipment     |
+---------+----------+--------------+

